In [0]:
%sql
select * from delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

order_id,customer_name,product,quantity,unit_price,order_date,status
1014,Eva Martinez1,Headphones,2,159.99,2024-02-14,Shipped
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered


In [0]:
%sql
describe detail delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,4ad6caa4-a7ae-4b06-a372-7286863bbee1,null,null,dbfs:/Volumes/useastws/default/deltavolume/ordersdata1,2026-08-26T15:07:37.447Z,2026-08-26T15:25:59Z,List(),List(),2,4927,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
describe extended delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

col_name,data_type,comment
order_id,int,null
customer_name,string,null
product,string,null
quantity,int,null
unit_price,float,null
order_date,string,null
status,string,null
,,
# Delta Statistics Columns,,
Column Names,"customer_name, unit_price, product, quantity, order_id, order_date, status",


In [0]:
%sql
create or replace temp view bad_recs as    
select * from values
(1003, 'Carol White',   'Monitor',    'three',  389.99, ('2024-02-02'),  'Delivered'),
(1012, 'Eva Martinez',  'Headphones',  'one',  159.99, ('2024-02-14'), 'Shipped')
as t(order_id , customer_name,product, quantity, unit_price,order_date, status);

schema enforcement

In [0]:
%sql
insert into delta.`/Volumes/useastws/default/deltavolume/ordersdata1`
select * from bad_recs;

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-4545980698230883>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'insert into delta.`/Volumes/useastws/default/deltavolume/ordersdata1`\nselect * from bad_recs;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:130, in SqlMagic.sql(self, line, cell)
    126     raise 

In [0]:
%sql
create or replace temp view bad_recs as    
select * from values
(1003, 'Carol White',   'Monitor',    '2',  389.99, ('2024-02-02'),  'Delivered'),
(1012, 'Eva Martinez',  'Headphones',  '1',  159.99, ('2024-02-14'), 'Shipped')
as t(order_id , customer_name,product, quantity, unit_price,order_date, status);

In [0]:
%sql
insert into delta.`/Volumes/useastws/default/deltavolume/ordersdata1`
select * from bad_recs;

num_affected_rows,num_inserted_rows
2,2


string casted to int if possible - '2' -> 2

In [0]:
%sql
select * from delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

order_id,customer_name,product,quantity,unit_price,order_date,status
1014,Eva Martinez1,Headphones,2,159.99,2024-02-14,Shipped
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered
1003,Carol White,Monitor,2,389.99,2024-02-02,Delivered
1012,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped


In [0]:
%sql
create or replace temp view add_col as    
select * from values
(1015, 'Carol White',   'Monitor',    '2',  389.99, ('2024-02-02'),  'Delivered',10),
(1014, 'Eva Martinez',  'Headphones',  '1',  159.99, ('2024-02-14'), 'Shipped',5)
as t(order_id , customer_name,product, quantity, unit_price,order_date, status,discount);

In [0]:
%sql
insert into delta.`/Volumes/useastws/default/deltavolume/ordersdata1`
select * from add_col;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4545980698230890>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'insert into delta.`/Volumes/useastws/default/deltavolume/ordersdata1`\nselect * from add_col;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:130, in SqlMagic.sql(self, line, cell)
    126     raise E

Schema mismatch - error

In [0]:
%sql
alter table delta.`/Volumes/useastws/default/deltavolume/ordersdata1`
add column discount int

In [0]:
%sql
insert into delta.`/Volumes/useastws/default/deltavolume/ordersdata1`
select * from add_col;

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
describe history delta.`/Volumes/useastws/default/deltavolume/ordersdata1`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-08-26T16:59:48Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2513891002461586),0826-063441-ydade36q,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 2672)",null,Databricks-Runtime/16.4.x-scala2.13
7,2026-08-26T16:59:43Z,147836707444603,anooptu@gmail.com,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""discount"",""type"":""integer"",""nullable"":true,""metadata"":{}}}])",null,List(2513891002461586),0826-063441-ydade36q,6,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
6,2026-08-26T16:55:51Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2513891002461586),0826-063441-ydade36q,5,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 2432)",null,Databricks-Runtime/16.4.x-scala2.13
5,2026-08-26T15:25:59Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#7489 = order_id#7186)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [{""actionType"":""delete""}], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 1, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4927, numTargetBytesRemoved -> 4920, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 2811, materializeSourceTimeMs -> 96, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 964, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 1, rewriteTimeMs -> 1707)",null,Databricks-Runtime/16.4.x-scala2.13
4,2026-08-26T15:08:41Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#5109 = order_id#5095)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [{""actionType"":""delete""}], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 9, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4920, numTargetBytesRemoved -> 5405, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3534, materializeSourceTimeMs -> 158, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1327, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 9, rewriteTimeMs -> 1956)",null,Databricks-Runtime/16.4.x-scala2.13
3,2026-08-26T15:08:29Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#3586 = order_id#3571)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""is_deleted#3578: boolean"",""actionType"":""delete""},{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(474475763794151),0826-063441-ydade36q,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 1, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2453, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4395, mater

In [0]:
%sql
select * from delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

order_id,customer_name,product,quantity,unit_price,order_date,status,discount
1015,Carol White,Monitor,2,389.99,2024-02-02,Delivered,10
1014,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped,5
1014,Eva Martinez1,Headphones,2,159.99,2024-02-14,Shipped,null
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered,null
1003,Carol White,Monitor,2,389.99,2024-02-02,Delivered,null
1012,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped,null


In [0]:
%sql
create or replace temp view new_batch as    
select * from values
(1015, 'Carol White',   'Monitor',    '2',  389.99, ('2024-02-02'),  'Delivered',10,'PROMO1'),
(1016, 'Eva Martinez',  'Headphones',  '1',  159.99, ('2024-02-14'), 'Shipped',5,'PROMO2')
as t(order_id , customer_name,product, quantity, unit_price,order_date, status,discount,promocode);

In [0]:
%sql
merge with schema evolution into delta.`/Volumes/useastws/default/deltavolume/ordersdata1` as target
using new_batch as src
on src.order_id = target.order_id
when matched then update set *
when not matched then insert *;


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
select * from delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

order_id,customer_name,product,quantity,unit_price,order_date,status,discount,promocode
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered,null,null
1014,Eva Martinez1,Headphones,2,159.99,2024-02-14,Shipped,null,null
1003,Carol White,Monitor,2,389.99,2024-02-02,Delivered,null,null
1012,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped,null,null
1014,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped,5,null
1015,Carol White,Monitor,2,389.99,2024-02-02,Delivered,10,PROMO1
1016,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped,5,PROMO2


same thing can be done using DF syntax as below
df.write\
.format('delta')\
.mode('append') \
.option('mergeSchema','true')\
.save('path_to_delta_table')

In [0]:
%sql
show tblproperties delta.`/Volumes/useastws/default/deltavolume/ordersdata1`

key,value
delta.enableDeletionVectors,true
delta.feature.appendOnly,supported
delta.feature.deletionVectors,supported
delta.feature.invariants,supported
delta.minReaderVersion,3
delta.minWriterVersion,7


In [0]:
from decimal import Decimal
from pyspark.sql.types import *
schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("product", StringType()),
    StructField("quantity", LongType()),
    StructField("unit_price", FloatType()),
    StructField("order_date", StringType()),
    StructField("status", StringType()),
    StructField("promo", StringType())
])

In [0]:
data = [
       (1017, 'Alice Johnson',  'Laptop',         1, 1299.99, '2024-01-15', 'Delivered','PROMO1'),
    (1018, 'Bob Smith',     'Wireless Mouse',  3,   29.99, '2024-01-18', 'Delivered','PROMO2'),
]

In [0]:
df = spark.createDataFrame(data, schema=schema)

In [0]:
display(df.printSchema())

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: float (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- promo: string (nullable = true)



source has 'quantity' as long type. But on target delta table the same column is only int. 

In [0]:
df.write\
    .mode('append')\
        .format('delta')\
            .option('mergeSchema','true')\
                .save('/Volumes/useastws/default/deltavolume/ordersdata1')

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4545980698230908>, line 5
      1 df.write\
      2     .mode('append')\
      3         .format('delta')\
      4             .option('mergeSchema','true')\
----> 5                 .save('/Volumes/useastws/default/deltavolume/ordersdata1')

File /databricks/spark/python/pyspark/databricks/instrumentation/instrumentation_utils.py:217, in _wrap_function.<locals>.wrapper(*args, **kwargs)
    215 start = time.perf_counter()
    216 try:
--> 217     res = func(*args, **kwargs)
    218     logging_helper.log_event(
    219         accessor=wrapper,
    220         module_name=module_name,
   (...)
    224         duration=time.perf_counter() - start
    225     )
    226     return res

File /databricks/spark/python/pyspark/sql/connect/readwriter.py:679, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)


Below property if set on table,will allow the type widening, like int-> float etc  'enableTypeWidening'

In [0]:
%sql
alter table delta.`/Volumes/useastws/default/deltavolume/ordersdata1`
set tblproperties ('delta.enableTypeWidening' = 'true')


In [0]:
df.write\
    .mode('append')\
        .format('delta')\
            .option('mergeSchema','true')\
                .save('/Volumes/useastws/default/deltavolume/ordersdata1')

In [0]:
%sql
select * from delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

order_id,customer_name,product,quantity,unit_price,order_date,status,discount,promocode,promo
1003,Carol White,Monitor,3,389.99,2024-02-02,Delivered,null,null,null
1014,Eva Martinez1,Headphones,2,159.99,2024-02-14,Shipped,null,null,null
1003,Carol White,Monitor,2,389.99,2024-02-02,Delivered,null,null,null
1012,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped,null,null,null
1014,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped,5,null,null
1015,Carol White,Monitor,2,389.99,2024-02-02,Delivered,10,PROMO1,null
1016,Eva Martinez,Headphones,1,159.99,2024-02-14,Shipped,5,PROMO2,null
1017,Alice Johnson,Laptop,1,1299.99,2024-01-15,Delivered,null,null,PROMO1
1018,Bob Smith,Wireless Mouse,3,29.99,2024-01-18,Delivered,null,null,PROMO2


In [0]:
%sql
describe extended delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

col_name,data_type,comment
order_id,int,null
customer_name,string,null
product,string,null
quantity,bigint,null
unit_price,float,null
order_date,string,null
status,string,null
discount,int,null
promocode,string,null
promo,string,null


Quantity column changed to Bigint which is same as float

In [0]:
%sql
describe history delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
14,2026-08-26T17:33:30Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2513891002461586),0826-063441-ydade36q,13,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 2760)",null,Databricks-Runtime/16.4.x-scala2.13
13,2026-08-26T17:33:18Z,147836707444603,anooptu@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableTypeWidening"":""true""})",null,List(2513891002461586),0826-063441-ydade36q,12,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
12,2026-08-26T17:31:23Z,147836707444603,anooptu@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableTypeWidening"":""false""})",null,List(2513891002461586),0826-063441-ydade36q,11,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
11,2026-08-26T17:19:08Z,147836707444603,anooptu@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableTypeWidening"":""true""})",null,List(2513891002461586),0826-063441-ydade36q,10,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
10,2026-08-26T17:05:36Z,147836707444603,anooptu@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2513891002461586),0826-063441-ydade36q,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 6, numRemovedBytes -> 15946, p25FileSize -> 3255, numDeletionVectorsRemoved -> 1, minFileSize -> 3255, numAddedFiles -> 1, maxFileSize -> 3255, p75FileSize -> 3255, p50FileSize -> 3255, numAddedBytes -> 3255)",null,Databricks-Runtime/16.4.x-scala2.13
9,2026-08-26T17:05:29Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#3101 = order_id#3118)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2513891002461586),0826-063441-ydade36q,8,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 5915, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 6299, materializeSourceTimeMs -> 204, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2775, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3221)",null,Databricks-Runtime/16.4.x-scala2.13
8,2026-08-26T16:59:48Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2513891002461586),0826-063441-ydade36q,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 2672)",null,Databricks-Runtime/16.4.x-scala2.13
7,2026-08-26T16:59:43Z,147836707444603,anooptu@gmail.com,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""discount"",""type"":""integer"",""nullable"":true,""metadata"":{}}}])",null,List(2513891002461586),0826-063441-ydade36q,6,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
6,2026-08-26T16:55:51Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2513891002461586),0826-063441-ydade36q,5,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 2432)",null,Databricks-Runtime/16.4.x-scala2.13
5,2026-08-26T15:25:59Z,147836707444603,anooptu@gmail.com,MERGE,"Map(predicate -> [""(order_id#7489 = order_id#7186)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [